# Public portfolio notebook
Outputs have been removed before publication. Execution requires authorised local course data. Saved aggregate results are in `reports/portfolio/`; the public dashboard uses synthetic data.


# BNPL feature quality v3

All 46 predictors have a formula, unit, missing-value rule and availability assumption. This
notebook reports development-only checks; it never opens test feature or label values.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/analysis_v1.json').exists())
sys.path.insert(0, str(root))
from src.bnpl_analysis.common import load_config
from src.run_artifacts import snapshot_manifest_path
config = load_config(root)
pd.set_option('display.max_columns', 12)


In [ ]:
from src.bnpl_analysis.feature_qa import run_feature_qa
qa_dir = root / 'reports/feature_qa' / config['snapshot_run_id']
if not (qa_dir / 'summary.json').exists():
    run_feature_qa(root)
quality = json.loads((qa_dir / 'summary.json').read_text(encoding='utf-8'))
assert quality['status'] == 'PASS'
assert quality['test_feature_or_label_values_read'] is False
print('Predictors checked:', quality['predictor_count'])
print('Development rows:', quality['development_rows'])
display(pd.DataFrame(quality['coverage_by_cutoff']))


## Missingness and stability
Coverage is observed evidence, not a measured fraud rate. Do not drop correlated predictors automatically.

In [ ]:
drift = pd.DataFrame(quality['train_validation_drift'])
drift['absolute_change'] = drift['missingness_change_pp'].abs()
display(drift.sort_values('absolute_change', ascending=False).head(10))
print('Complete per-cutoff profiles:', len(quality['profiles']))
print('Near-constant feature/date combinations:', len(quality['near_constant_features']))


## Interpretation
Consumer latest now averages distinct users on the latest observed signal date. Sparse merchant fraud and changing consumer coverage warrant the no-fraud comparison. See the versioned feature catalogue for all formulas; final test performance is still unevaluated.